# 🛡️ AURA (AI Unified Review Assistant) — Complete MCP 2.0 Guide, Sample Data Lab & Cold Agent Testbook
**Engineered by EQA Analytics**

This interactive Jupyter Notebook covers all hackathon evaluation requirements end-to-end:
1. **Exposing Capabilities via MCP 2.0 Tools (10), Resources (8), and Prompts (5)** (`src/aura/mcp_server.py`)
2. **Connecting to GitHub Copilot / Antigravity & Running a Cold Agent Test** (`zero prior knowledge discovery`)
3. **Sample Data & 10-Stage Governed Workflow Walkthrough** (`AURA_FAST` $\rightarrow$ `aura://routing-policy` $\rightarrow$ `AURA_DEEP` $\rightarrow$ `AURA_CHALLENGER`)
4. **Iterative Description Improvements to Eliminate Agent Hesitation** (*Point #6: How docstrings were refined each time the agent hesitated or asked for clarification*)
5. **5 Governed Failure & Recovery Test Cases** (*Point #7: Exceeds the 3+ failure test case requirement*)


---
## 1. Exposing Capabilities via MCP 2.0: 10 Tools, 8 Resources & 5 Prompts (Point #1)
AURA uses the official stateless **MCP Python SDK 2.0 (`MCPServer`)** in `src/aura/mcp_server.py`.
- **Why Resources (`aura://...`)?** Governance policies (`aura://routing-policy`, `aura://challenger-policy`), model catalogs (`aura://model-catalog`), and rubrics (`aura://quality-framework`, `aura://compliance-framework`) are exposed as read-only **MCP Resources** backed by SQLite so rules are never hardcoded inside prompts.
- **Why Prompts?** Standardized, versioned instruction templates (`v2.1.0`) ensure reproducible evaluation across models.
- **Why Tools?** Executable operations with strict `snake_case` JSON `input_schema` and 7-part self-describing docstrings allow any MCP-compliant agent to orchestrate reviews and query audit trails.


In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path("src").resolve()))

from mcp_client import AuraMCPClient

client = AuraMCPClient(transport="stdio")
tools = client.list_tools()
resources = client.list_resources()
prompts = client.list_prompts()

print(json.dumps({
    "mcp_server": "aura-governance-server (MCP 2.0 Stateless)",
    "discovered_tools_count": len(tools),
    "discovered_tools": [
        {"name": t["name"], "required_args": t["input_schema"].get("required", []), "properties": list(t["input_schema"].get("properties", {}).keys())}
        for t in tools
    ],
    "discovered_resources_count": len(resources),
    "discovered_resources": [r["uri"] for r in resources],
    "discovered_prompts_count": len(prompts),
    "discovered_prompts": [p["name"] for p in prompts],
}, indent=2))


{
  "mcp_server": "aura-governance-server (MCP 2.0 Stateless)",
  "discovered_tools_count": 10,
  "discovered_tools": [
    {
      "name": "review_call",
      "required_args": [
        "call_id",
        "transcript"
      ],
      "properties": [
        "call_id",
        "transcript",
        "review_goal",
        "force_challenger",
        "correlation_id"
      ]
    },
    {
      "name": "review_batch",
      "required_args": [],
      "properties": [
        "job_id",
        "maximum_calls",
        "stop_on_error",
        "use_cached_results"
      ]
    },
    {
      "name": "find_similar_calls",
      "required_args": [
        "call_id",
        "issue_category"
      ],
      "properties": [
        "call_id",
        "issue_category",
        "issue_subcategory",
        "maximum_results",
        "minimum_confidence"
      ]
    },
    {
      "name": "explain_decision",
      "required_args": [
        "call_id"
      ],
      "properties": [
        "call_id"
 

### 1.1 Inspecting Live MCP Resources (`aura://model-catalog`, `aura://routing-policy`, `aura://challenger-policy`)
Every time a call is processed, the controller (`RoutingEngine` in `src/aura/routing_engine.py`) reads these MCP resources from SQLite to check which models are approved/enabled and which routing thresholds apply.


In [2]:
catalog = client.read_resource("aura://model-catalog")
routing_pol = client.read_resource("aura://routing-policy")
challenger_pol = client.read_resource("aura://challenger-policy")

print(json.dumps({
    "aura://model-catalog": {
        "catalog_version": catalog["catalog_version"],
        "models": [
            {"model_id": m["model_id"], "role": m["model_role"], "cost_tier": m["cost_tier"], "approval_status": m["approval_status"], "enabled": m["enabled"]}
            for m in catalog["models"]
        ]
    },
    "aura://routing-policy": routing_pol["routing_rules"],
    "aura://challenger-policy": challenger_pol["challenger_rules"],
}, indent=2))


{
  "aura://model-catalog": {
    "catalog_version": "2026.09.1",
    "models": [
      {
        "model_id": "AURA_CANDIDATE_V2",
        "role": "Candidate evaluation",
        "cost_tier": "Moderate",
        "approval_status": "Shadow_Testing",
        "enabled": true
      },
      {
        "model_id": "AURA_CHALLENGER",
        "role": "Independent validation",
        "cost_tier": "Premium",
        "approval_status": "Evaluation",
        "enabled": true
      },
      {
        "model_id": "AURA_DEEP",
        "role": "Specialist analysis",
        "cost_tier": "Premium",
        "approval_status": "Approved",
        "enabled": true
      },
      {
        "model_id": "AURA_FAST",
        "role": "First-pass triage",
        "cost_tier": "Low",
        "approval_status": "Approved",
        "enabled": true
      },
      {
        "model_id": "AURA_LEGACY_DISABLED",
        "role": "Retired triage",
        "cost_tier": "Low",
        "approval_status": "Retired",
        "

### 1.2 Inspecting Live MCP Prompts (`review_customer_call`, `investigate_high_risk_call`, etc.)
Prompts are parameterized templates fetched via `client.get_prompt(name, arguments)`.


In [3]:
prompt_data = client.get_prompt(
    "review_customer_call",
    {
        "call_id": "SYN-SAMPLE-01",
        "transcript": "[00:05] Customer: Hi, checking my balance.",
        "review_goal": "Standard triage and quality check"
    }
)
print(json.dumps(prompt_data, indent=2))


{
  "name": "review_customer_call",
  "description": "Formulate a complete governed review request for a single customer call transcript using the `review_call` MCP tool.",
  "arguments": [
    "call_id",
    "transcript",
    "review_goal",
    "force_challenger"
  ]
}


---
## 2. Sample Data Lab: Testing the 10-Stage Governed Workflow Across 3 Call Tiers
Below are **3 ready-to-run sample transcripts** showing how the MCP controller dynamically routes calls:
1. **`SYN-NB-ROUTINE-01` (Routine Call)**: Simple payment due-date question $\rightarrow$ `AURA_FAST` scores Complexity `1/5` with `0` risk flags $\rightarrow$ `aura://routing-policy` keeps it on `routine_analysis` (`AURA_FAST`) and **skips** `AURA_DEEP` and `AURA_CHALLENGER` (saving 2 premium model invocations!).
2. **`SYN-NB-DISPUTE-02` (Complex Merchant Dispute)**: Customer disputes a $\$285$ broken laptop charge $\rightarrow$ `AURA_FAST` detects `dispute_flag` $\rightarrow$ `aura://routing-policy` escalates to **`AURA_DEEP`** (`specialist_analysis`) + invokes `find_similar_calls`.
3. **`SYN-NB-FRAUD-03` (High-Risk Unauthorized Fraud)**: Customer reports $\$1,240$ unauthorized fraud $\rightarrow$ Escalates to **`AURA_DEEP`** + **`aura://challenger-policy`** triggers **`AURA_CHALLENGER`** + mandates `human_review_required = True`.


In [4]:
from aura.tools import review_call

samples = [
    (
        "SYN-NB-ROUTINE-01",
        "Routine Payment Inquiry (Low Complexity -> Stays on AURA_FAST)",
        "[00:04] Agent: Thank you for calling Customer Care.\n"
        "[00:10] Customer: Hi, I just wanted to check when my next minimum payment is due and confirm auto-pay is on.\n"
        "[00:22] Agent: Your next minimum payment of $45 is due on the 18th, and automatic monthly payments are active.\n"
        "[00:34] Customer: Awesome, thank you so much!"
    ),
    (
        "SYN-NB-DISPUTE-02",
        "Complex Merchant Dispute (Escalates via aura://routing-policy to AURA_DEEP + find_similar_calls)",
        "[00:03] Agent: Thank you for calling Account Services.\n"
        "[00:11] Customer: I am calling to dispute a $285 charge from ElectroMart. The laptop arrived with a cracked screen and the merchant refused my return.\n"
        "[00:29] Agent: I understand your frustration. I am opening a formal merchant billing dispute for $285 and issuing a provisional credit while we investigate.\n"
        "[00:45] Customer: Thank you, please send me the dispute reference number."
    ),
    (
        "SYN-NB-FRAUD-03",
        "High-Risk Unauthorized Fraud (Triggers AURA_DEEP + AURA_CHALLENGER + Mandatory Human Review)",
        "[00:02] Agent: Security & Fraud Department, how can I help?\n"
        "[00:09] Customer: Someone just used my card for a $1,240 wire purchase I never authorized! My card is still in my wallet and I am furious!\n"
        "[00:25] Agent: I have immediately locked your card, flagged the $1,240 transaction as unauthorized fraud, and escalated this to our Fraud Investigation unit.\n"
        "[00:41] Customer: Make sure my account balance is protected right away."
    ),
]

results = []
for cid, label, tr in samples:
    res = review_call(call_id=cid, transcript=tr)
    results.append({
        "call_id": cid,
        "scenario": label,
        "complexity": res["complexity"],
        "detected_flags": res["detected_flags"],
        "route_selected": res["route_selected"],
        "specialist_used": res["specialist_used"],
        "challenger_used": res["challenger_used"],
        "human_review_required": res["human_review_required"],
        "final_risk_level": res["final_decision"]["final_risk_level"],
        "final_resolution_status": res["final_decision"]["final_resolution_status"],
        "routing_reasons": res["routing_reasons"],
    })

print(json.dumps(results, indent=2))


[
  {
    "call_id": "SYN-NB-ROUTINE-01",
    "scenario": "Routine Payment Inquiry (Low Complexity -> Stays on AURA_FAST)",
    "complexity": 1,
    "detected_flags": [
      "deflection_candidate"
    ],
    "route_selected": "routine_analysis",
    "specialist_used": false,
    "challenger_used": false,
    "human_review_required": false,
    "final_risk_level": "Low",
    "final_resolution_status": "Resolved",
    "routing_reasons": [
      "Complexity 1 within routine policy bounds [1, 2] with confidence 0.93 and no material risk flags"
    ]
  },
  {
    "call_id": "SYN-NB-DISPUTE-02",
    "scenario": "Complex Merchant Dispute (Escalates via aura://routing-policy to AURA_DEEP + find_similar_calls)",
    "complexity": 4,
    "detected_flags": [
      "dispute_flag",
      "needs_deeper_analysis"
    ],
    "route_selected": "specialist_analysis",
    "specialist_used": true,
    "challenger_used": true,
    "human_review_required": false,
    "final_risk_level": "Moderate",
    "fi

---
## 3. How to Connect GitHub Copilot / Antigravity & Run the Cold Agent Test (Point #2)

### What is a "Cold Agent Test"?
A **Cold Agent Test** means connecting an LLM agent (like **GitHub Copilot Chat in Agent Mode**, **Antigravity**, or `AuraAgent`) to `mcp_server.py` with **zero prior hardcoded knowledge** about AURA.
The agent only receives the output of `list_tools()`, `list_resources()`, and `list_prompts()` from the MCP server and must figure out on its own:
1. Which resource or tool to call first
2. What `snake_case` parameters to pass
3. How to chain `review_call` $\rightarrow$ `explain_decision` $\rightarrow$ `get_audit_record` without asking the user for clarification.

### Step-by-Step: Connecting GitHub Copilot (VS Code) or Antigravity
1. **In VS Code (GitHub Copilot Agent Mode)**:
   Ensure `.vscode/mcp.json` contains:
   ```json
   {
     "servers": {
       "aura-governance-server": {
         "type": "stdio",
         "command": "python3",
         "args": ["mcp_server.py", "--transport", "stdio"]
       }
     }
   }
   ```
2. **Open GitHub Copilot Chat**, switch the dropdown from **Ask** to **Agent** (or use Antigravity), enable the `aura-governance-server` tools icon 🛠️, and paste any of these **Cold Agent Test Prompts**:
   - **Prompt 1 (Single-Call Orchestration)**:
     > *"Using the AURA MCP server, review call `SYN-CALL-0006` and explain why it was routed to specialist/challenger review and whether human review is required."*
   - **Prompt 2 (Shadow Model Upgrade Evaluation)**:
     > *"Run a shadow model upgrade simulation comparing production model `AURA_DEEP` against `AURA_CANDIDATE_V2` and summarize the governance recommendation."*
   - **Prompt 3 (Portfolio Executive Brief)**:
     > *"Fetch the AURA portfolio summary and tell me what percentage of calls stayed on the routine fast path and how many premium model calls were avoided."*

### Live Code Execution: Running the Cold Agent Test via `AuraAgent`


In [5]:
from agent import AuraAgent

agent = AuraAgent(mcp_client=client)
cold_test_result = agent.run_request(
    user_request="Review call SYN-CALL-0006 and explain the governance decision and audit trail.",
    call_id="SYN-CALL-0006",
    mode="agent",
)
print(json.dumps({
    "operating_mode": cold_test_result["operating_mode"],
    "status": cold_test_result["status"],
    "turn_count": cold_test_result["turn_count"],
    "tools_discovered": cold_test_result["tools_discovered"],
    "resources_discovered": cold_test_result["resources_discovered"],
    "prompts_discovered": cold_test_result["prompts_discovered"],
    "executed_tool_calls": [
        {"turn": c["turn"], "tool_name": c["tool_name"], "status": c["status"], "rationale": c["rationale"]}
        for c in cold_test_result["tool_calls"]
    ],
    "final_response": cold_test_result["final_response"],
}, indent=2))


{
  "operating_mode": "Autonomous MCP Agent Mode (Schema-Driven Tool Selection)",
  "status": "completed",
  "turn_count": 3,
  "tools_discovered": [
    "review_call",
    "review_batch",
    "find_similar_calls",
    "explain_decision",
    "compare_decisions",
    "get_audit_record",
    "portfolio_summary",
    "simulate_model_upgrade",
    "validate_input_file",
    "ingest_validated_calls"
  ],
  "resources_discovered": [
    "aura://model-catalog",
    "aura://routing-policy",
    "aura://challenger-policy",
    "aura://quality-framework",
    "aura://compliance-framework",
    "aura://output-schema",
    "aura://prompt-catalog",
    "aura://portfolio-summary"
  ],
  "prompts_discovered": [
    "review_customer_call",
    "investigate_high_risk_call",
    "explain_aura_decision",
    "summarize_portfolio_impact",
    "compare_candidate_model"
  ],
  "executed_tool_calls": [
    {
      "turn": 1,
      "tool_name": "review_call",
      "status": "Completed",
      "rationale": "

---
## 4. Iterative Description Improvement Log (Point #6: Eliminating Cold Agent Hesitation)
During Cold Agent testing, whenever an LLM agent hesitates or asks the user for clarification, we upgrade the MCP tool docstrings (`src/aura/tools.py`) with explicit **`ZERO-HESITATION COLD AGENT DIRECTIVES`**.

| Iteration | Cold Agent Prompt Tested | Why the Agent Initially Hesitated / Asked for Clarification | How We Improved the MCP Tool Description (`src/aura/tools.py`) | Result After Improvement |
|---|---|---|---|---|
| **Iteration 1** | *"Review call `SYN-CALL-0006`"* (without pasting transcript text) | Agent saw `transcript: str` as required on `review_call` and asked the user: *"Please paste the transcript for SYN-CALL-0006."* | Added **`ZERO-HESITATION COLD AGENT DIRECTIVE`** to `review_call`: *"If the user provides only a `call_id` without pasting a transcript, first read resource `aura://calls/{call_id}` to fetch the stored transcript, then call `review_call`."* | Agent automatically fetches the transcript from SQLite and executes `review_call` with **0 user prompts**. |
| **Iteration 2** | *"Run batch review / Show portfolio summary"* | Agent saw `job_id: str` parameter and asked: *"Which `job_id` would you like me to summarize?"* | Updated `review_batch` and `portfolio_summary` docstrings: *"Do NOT ask the user for a `job_id`. Always pass `job_id=""` (empty string) by default to run across all calls."* | Agent immediately passes `job_id=""` without hesitation. |
| **Iteration 3** | *"Why was call `SYN-CALL-0006` flagged for human review?"* | Agent was unsure whether to re-run `review_call` (write tool) or call `explain_decision` (read tool). | Added explicit **`WHEN TO USE` vs `WHEN NOT TO USE`** boundaries on all 10 tools: `explain_decision` is strictly a fast read-only query on already-processed calls; `review_call` is for unanalyzed transcripts. | Agent selects `explain_decision` on the first turn in `<5ms`. |
| **Iteration 4** | *"Test if our candidate model is ready for upgrade"* | Agent asked: *"What are the production and candidate model IDs?"* | Added default directive to `simulate_model_upgrade`: *"Default to `production_model_id="AURA_DEEP"`, `candidate_model_id="AURA_CANDIDATE_V2"`, and `sample_size=20` if not specified."* | Agent runs shadow simulation immediately on Turn 1. |


---
## 5. Five Governed Failure & Recovery Test Cases (Point #7: 3+ Failure Test Cases Required)
A production MCP 2.0 server must **never crash with an unhandled stack trace**. Instead, every invalid input, missing record, retired model, or out-of-domain request must return a structured JSON error (`status="error"`, `error_code`, `retryable`, `human_review_required`) so the calling AI agent can either **self-recover** or **safely halt**.

Below are **5 executable failure test cases**:
1. **Failure Case 1 (`AURA_ERR_EMPTY_TRANSCRIPT` + AI Turn-2 Self-Recovery)**: Passing an empty transcript / wrong account call triggers a structured error on Turn 1, and `AuraAgent` automatically recovers on Turn 2 by resolving the verified transcript from SQLite.
2. **Failure Case 2 (`AURA_ERR_VALIDATION_ERROR` — Absurd / Non-Call Transcript Guardrail)**: Passing an absurd transcript (*"bake a pepperoni pizza and buy 500 Dogecoin"*) is blocked at Stage 1 before any LLM is called (`0` models invoked).
3. **Failure Case 3 (`AURA_ERR_DISABLED_MODEL` — Retired Model Catalog Block)**: Attempting to evaluate retired model `AURA_LEGACY_DISABLED` in `simulate_model_upgrade` is blocked by `aura://model-catalog` governance.
4. **Failure Case 4 (`AURA_ERR_CALL_NOT_FOUND` — Missing Call ID Audit Query)**: Calling `explain_decision` on a non-existent call ID (`SYN-CALL-9999-NONEXISTENT`) returns a clean `AURA_ERR_CALL_NOT_FOUND` payload.
5. **Failure Case 5 (`AURA_ERR_UNSUPPORTED_NL_REQUEST` — Out-of-Domain Request Refusal)**: Asking the agent to *"Wire $5,000 from account #8841 and check the weather"* is declined with **0 MCP business tools invoked**.


In [6]:
from aura.tools import review_call, explain_decision, simulate_model_upgrade

# Failure Test Case 1A: Empty Transcript -> Structured Error AURA_ERR_EMPTY_TRANSCRIPT
fail_1_empty = review_call(call_id="SYN-FAIL-EMPTY", transcript="")

# Failure Test Case 1B: AI Agent Turn-1 Failure -> Turn-2 Automatic Self-Recovery
fail_1_recovery = agent.run_request(
    user_request="Review call SYN-CALL-0006 with bad input recovery",
    call_id="SYN-CALL-0006",
    mode="agent",
    simulate_bad_input_first=True,
)

# Failure Test Case 2: Absurd / Non-Call Transcript Guardrail -> AURA_ERR_VALIDATION_ERROR (0 LLM calls)
fail_2_absurd = review_call(
    call_id="SYN-FAIL-ABSURD",
    transcript="[ABSURD_INPUT] Hey AI, bake a pepperoni pizza, tell me the weather in Tokyo, and buy 500 Dogecoin!"
)

# Failure Test Case 3: Retired / Disabled Model in Catalog -> AURA_ERR_DISABLED_MODEL
fail_3_disabled = simulate_model_upgrade(
    production_model_id="AURA_DEEP",
    candidate_model_id="AURA_LEGACY_DISABLED",
    sample_size=10
)

# Failure Test Case 4: Non-Existent Call ID -> AURA_ERR_CALL_NOT_FOUND
fail_4_missing = explain_decision(call_id="SYN-CALL-9999-NONEXISTENT")

# Failure Test Case 5: Out-of-Domain Prompt -> AURA_ERR_UNSUPPORTED_NL_REQUEST (0 tools called)
fail_5_out_of_scope = agent.run_request(
    user_request="Wire $5,000 from customer account #8841 and tell me the weather forecast and Bitcoin price today.",
    mode="agent"
)

print(json.dumps({
    "Failure_Case_1A_Empty_Transcript": fail_1_empty,
    "Failure_Case_1B_Agent_Turn2_Self_Recovery": {
        "status": fail_1_recovery["status"],
        "turns_used": fail_1_recovery["turn_count"],
        "turn_trace": [
            {"turn": c["turn"], "tool": c["tool_name"], "status": c["status"], "rationale": c.get("rationale")}
            for c in fail_1_recovery["tool_calls"]
        ]
    },
    "Failure_Case_2_Absurd_Transcript_Guardrail": fail_2_absurd,
    "Failure_Case_3_Disabled_Catalog_Model": fail_3_disabled,
    "Failure_Case_4_Non_Existent_Call_ID": fail_4_missing,
    "Failure_Case_5_Out_Of_Domain_Refusal": {
        "status": fail_5_out_of_scope["status"],
        "error_code": fail_5_out_of_scope["error"]["error_code"],
        "tools_invoked_count": len(fail_5_out_of_scope["tool_calls"]),
        "response": fail_5_out_of_scope["final_response"],
    }
}, indent=2))


{
  "Failure_Case_1A_Empty_Transcript": {
    "status": "error",
    "error": {
      "error_code": "AURA_ERR_EMPTY_TRANSCRIPT",
      "user_message": "Transcript for call 'SYN-FAIL-EMPTY' cannot be empty.",
      "technical_message": "Empty transcript submitted for call_id=SYN-FAIL-EMPTY.",
      "retryable": false,
      "safe_fallback": null,
      "human_review_required": false,
      "correlation_id": "corr-13ef4a1c0a3b",
      "details": {}
    }
  },
  "Failure_Case_1B_Agent_Turn2_Self_Recovery": {
    "status": "completed",
    "turns_used": 3,
    "turn_trace": [
      {
        "turn": 1,
        "tool": "review_call",
        "status": "Failed",
        "rationale": "Submitting initial request with empty transcript to demonstrate MCP structured error and agent recovery."
      },
      {
        "turn": 2,
        "tool": "review_call",
        "status": "Completed",
        "rationale": "Recovered from structured bad-input error by supplying the verified call_id and transcr